In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
!pip uninstall -y torchvision -q
!pip install -q -U "transformers>=4.41" "accelerate" "bitsandbytes>=0.46.1" "peft" "torchao>=0.16.0" scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.26 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have s

In [ ]:
import os, json, time, re, gc
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
)
from datasets import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
USE_LORA = True    
RUN_MODE = "full"  

def find_dataset_csv(root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        for f in files:
            if f.endswith(".csv") and "suicide" in f.lower():
                return os.path.join(r, f)
    raise FileNotFoundError("CSV introuvable sous /kaggle/input")

RAW_CSV = find_dataset_csv()
print("CSV:", RAW_CSV)

OUT_DIR = "/kaggle/working"
RESULTS_DIR = os.path.join(OUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

TOTAL_SUBSET_SIZE = 15000
VAL_SIZE = 1500
MAX_LEN = 128

torch.cuda.empty_cache()
print(f"MODEL_NAME={MODEL_NAME}  USE_LORA={USE_LORA}  RUN_MODE={RUN_MODE}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


CSV: /kaggle/input/datasets/nikhileswarkomati/suicide-watch/Suicide_Detection.csv
MODEL_NAME=meta-llama/Llama-3.2-1B-Instruct  USE_LORA=True  RUN_MODE=full


In [5]:
df = pd.read_csv(RAW_CSV)
print("Taille brute:", df.shape)

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r"http\S+|www\.\S+", "", t)
    t = t.replace("[deleted]", "").replace("[removed]", "")
    t = re.sub(r"\s+", " ", t).strip()
    return t

text_col = "text" if "text" in df.columns else df.columns[1]
label_col = "class" if "class" in df.columns else df.columns[-1]

df["clean_text"] = df[text_col].apply(clean_text)
df = df[df["clean_text"].str.len() > 5]
df = df.drop_duplicates(subset="clean_text")
df["label"] = df[label_col].map({"suicide": 1, "non-suicide": 0}).astype(int)
df = df[["clean_text", "label"]].reset_index(drop=True)
print("Apres nettoyage:", df.shape)

Taille brute: (232074, 3)
Apres nettoyage: (231972, 2)


In [6]:
subset, _ = train_test_split(df, train_size=TOTAL_SUBSET_SIZE, stratify=df["label"], random_state=SEED)
train_pool, test_full = train_test_split(subset, test_size=0.20, stratify=subset["label"], random_state=SEED)
train_subsample, val_subsample = train_test_split(train_pool, test_size=VAL_SIZE, stratify=train_pool["label"], random_state=SEED)

print("train_subsample:", train_subsample.shape, " val_subsample:", val_subsample.shape, " test_full:", test_full.shape)

if RUN_MODE == "smoke":
    train_subsample_run = train_subsample.sample(200, random_state=SEED)
else:
    train_subsample_run = train_subsample
print("Lignes d'entrainement utilisees:", len(train_subsample_run))

train_subsample: (10500, 2)  val_subsample: (1500, 2)  test_full: (3000, 2)
Lignes d'entrainement utilisees: 10500


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    return tokenizer(batch["clean_text"], truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_subsample_run[["clean_text", "label"]]).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_subsample[["clean_text", "label"]]).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_full[["clean_text", "label"]]).map(tokenize_fn, batched=True)
train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")
collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/10500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,dtype=torch.float32)
model.config.pad_token_id = tokenizer.pad_token_id

if USE_LORA:
    from peft import LoraConfig, get_peft_model
    if "phi" in MODEL_NAME.lower():
        target_modules = ["q_proj", "k_proj", "v_proj", "dense"]
    else:
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
        
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        modules_to_save=["score"],   # critique -- voir FYP_Development_Log.md section 5
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
    )
    model = get_peft_model(model, lora_config)
    for name, param in model.named_parameters():
        if "modules_to_save" in name:
            param.requires_grad = True
    model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,708,032 || all params: 1,237,526,528 || trainable%: 0.1380


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

class_counts = train_subsample_run["label"].value_counts().sort_index().values
class_weights = torch.tensor(class_counts.sum() / (2.0 * class_counts), dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(device=logits.device, dtype=logits.dtype))
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [10]:
model_slug = MODEL_NAME.split("/")[-1]

training_args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, f"ckpt_{model_slug}"),
    eval_strategy="steps",
    eval_steps=10 if RUN_MODE == "smoke" else 200,
    save_strategy="steps",
    save_steps=10 if RUN_MODE == "smoke" else 200,
    save_total_limit=2,
    max_steps=50 if RUN_MODE == "smoke" else -1,
    learning_rate=2e-5 if not USE_LORA else 2e-4,   # LR plus haut usuel avec LoRA
    per_device_train_batch_size=4 if USE_LORA else 8,
    per_device_eval_batch_size=8 if USE_LORA else 16,
    gradient_accumulation_steps=4 if USE_LORA else 2,
    num_train_epochs=1 if RUN_MODE == "smoke" else 2,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",     # cle pour eviter l'OOM sur les modeles >1B
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5 if RUN_MODE == "smoke" else 50,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [11]:
t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print(f"Temps d'entrainement: {train_time_min:.1f} min")
if RUN_MODE == "smoke":
    print("Smoke test OK -> repassez RUN_MODE='full' en cellule 2 et relancez tout.")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
200,0.453900,0.102345,0.978667,0.994490,0.962667,0.978320
400,0.589719,0.060166,0.985333,0.995913,0.974667,0.985175
600,0.146853,0.074739,0.982000,0.976285,0.988000,0.982107
800,0.027481,0.078908,0.988000,0.997283,0.978667,0.987887
1000,0.030949,0.086637,0.983333,0.971391,0.996000,0.983542


Temps d'entrainement: 20.4 min


In [12]:
if RUN_MODE == "full":
    final_preds = trainer.predict(test_ds)
    logits, labels = final_preds.predictions, final_preds.label_ids
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    print(f"Accuracy={acc:.4f} Precision={p:.4f} Recall={r:.4f} F1={f1:.4f}")

    model.eval()
    sample_texts = test_full["clean_text"].sample(50, random_state=SEED).tolist()
    latencies = []
    with torch.no_grad():
        for t in sample_texts:
            inputs = tokenizer(t, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
            ts = time.time()
            _ = model(**inputs)
            torch.cuda.synchronize()
            latencies.append((time.time() - ts) * 1000)
    avg_latency_ms = float(np.mean(latencies))
    print(f"Latence moyenne: {avg_latency_ms:.2f} ms")

    result = {
        "model_name": MODEL_NAME, "model_slug": model_slug, "use_lora": USE_LORA,
        "n_train_samples": len(train_subsample_run), "n_test_samples": len(test_full),
        "accuracy": acc, "precision": p, "recall": r, "f1": f1,
        "train_time_min": train_time_min, "avg_inference_latency_ms": avg_latency_ms,
    }
    with open(os.path.join(RESULTS_DIR, f"{model_slug}.json"), "w") as f:
        json.dump(result, f, indent=2)

    final_model_dir = os.path.join(OUT_DIR, f"final_model_{model_slug}")
    if USE_LORA:
        model = model.merge_and_unload()   # fusionne les adaptateurs LoRA dans le modele de base
    trainer.save_model(final_model_dir) if not USE_LORA else model.save_pretrained(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)
    print(json.dumps(result, indent=2))
    print(f"Modele sauvegarde: {final_model_dir}")

Accuracy=0.9833 Precision=0.9885 Recall=0.9780 F1=0.9832
Latence moyenne: 45.52 ms


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model_name": "meta-llama/Llama-3.2-1B-Instruct",
  "model_slug": "Llama-3.2-1B-Instruct",
  "use_lora": true,
  "n_train_samples": 10500,
  "n_test_samples": 3000,
  "accuracy": 0.9833333333333333,
  "precision": 0.988544474393531,
  "recall": 0.978,
  "f1": 0.9832439678284183,
  "train_time_min": 20.425862165292106,
  "avg_inference_latency_ms": 45.51708698272705
}
Modele sauvegarde: /kaggle/working/final_model_Llama-3.2-1B-Instruct
